# Cleveland LVT Model

Models a 4:1 split-rate LVT for the City of Cleveland using Cuyahoga County parcel data.

**Tax system**: Ohio assesses at 35% of market value. City of Cleveland levies 12.70 mills total (4.40 unvoted + 8.30 charter). Source: City of Cleveland ACFR 2024, Note 10 and Statistical Section S8.

**Data**: Cuyahoga County MyPLACE GJOIN MapServer — geometry and CAMA attributes in one service.

**Column mapping**:
| Concept | Column | Notes |
|---|---|---|
| Land value (market) | `certified_tax_land` | After exemptions/abatements |
| Improvement value (market) | `certified_tax_building` | After exemptions/abatements |
| Total value (market) | `certified_tax_total` | After exemptions/abatements |
| Gross land (market) | `gross_certified_land` | Before exemptions |
| Gross improvement (market) | `gross_certified_building` | Before exemptions |
| Exempt indicator | `full_exmp` | 1 if certified_tax_total == 0 |
| Land use code | `tax_luc` | Drives property category |
| Property class | `property_class` | R/C/I/E/A/P/CE/IE/RE |
| Parcel ID | `parcelpin` | |

**Assessment ratio**: 35% (Ohio standard)
**City millage**: 12.70 mills
**Model type**: split_rate:4.0

In [ ]:
import sys
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely.geometry import Polygon, MultiPolygon
from dotenv import load_dotenv

matplotlib.use('Agg')

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'cleveland'
STATE_FIPS = '39'
COUNTY_FIPS = '035'
MODEL_TYPE = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0
ASSESSMENT_RATIO = 0.35
CITY_MILLAGE = 12.70

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Fetch / Load Parcel Data

In [ ]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'

BASE_URL = 'https://gis.cuyahogacounty.us/server/rest/services/MyPLACE/Parcels_WMA_GJOIN_WGS84/MapServer/2/query'
WHERE = "par_city='CLEVELAND'"

def rings_to_geometry(geom_data):
    if not geom_data or 'rings' not in geom_data:
        return None
    rings = geom_data['rings']
    if not rings:
        return None
    try:
        exterior = rings[0]
        holes = rings[1:] if len(rings) > 1 else []
        return Polygon(exterior, holes)
    except Exception:
        return None

def download_cleveland_parcels():
    count_r = requests.get(BASE_URL, params={'where': WHERE, 'returnCountOnly': 'true', 'f': 'json'}, timeout=30)
    total = count_r.json().get('count', 0)
    print(f'Total Cleveland parcels: {total:,}')

    all_features = []
    offset = 0
    chunk = 1000

    while offset < total:
        params = {
            'where': WHERE,
            'outFields': '*',
            'returnGeometry': 'true',
            'outSR': '4326',
            'geometryPrecision': 6,
            'resultOffset': offset,
            'resultRecordCount': chunk,
            'f': 'json',
        }
        r = requests.get(BASE_URL, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
        feats = data.get('features', [])
        if not feats:
            break
        for feat in feats:
            attrs = feat['attributes'].copy()
            attrs['geometry'] = rings_to_geometry(feat.get('geometry'))
            all_features.append(attrs)
        offset += len(feats)
        if offset % 10000 == 0:
            print(f'  Downloaded {offset:,} / {total:,}')
        if len(feats) < chunk:
            break

    gdf = gpd.GeoDataFrame(all_features, crs='EPSG:4326')
    print(f'Downloaded {len(gdf):,} parcels')
    return gdf

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f'Loaded {len(gdf):,} parcels from cache')
else:
    gdf = download_cleveland_parcels()
    gdf.to_parquet(PARCEL_PATH)
    print(f'Saved to {PARCEL_PATH}')

## Section 3 — Classify and Validate

In [ ]:
# Coerce value columns to numeric
val_cols = [
    'certified_tax_land', 'certified_tax_building', 'certified_tax_total',
    'certified_exempt_land', 'certified_exempt_building', 'certified_exempt_total',
    'certified_abated_land', 'certified_abated_building', 'certified_abated_total',
    'gross_certified_land', 'gross_certified_building', 'gross_certified_total',
]
for col in val_cols:
    gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0.0)

# Full exemption flag: certified_tax_total == 0
gdf['full_exmp'] = (gdf['certified_tax_total'] == 0).astype(int)

# Property category mapping
LARGE_APT_LUCS = {'4010', '4020', '4030', '4040', '4050', '4060', '4070', '4080'}
SMALL_APT_LUCS = {'4090'}  # 4-6 unit (commercial-rated, likely 5-6 units)
PARKING_LUCS = {'4560', '4565', '4566', '4567'}
COMM_VAC = {'4000'}
RES_VAC = {'5000'}

def classify_parcel(row):
    luc = str(row.get('tax_luc', '') or '').strip()
    cls = str(row.get('property_class', '') or '').strip()

    if cls == 'E':
        return 'Exempt'
    if cls == 'A' or luc.startswith('7'):
        return 'Agricultural'
    if cls == 'P' or luc.startswith('8'):
        return 'Other'
    if luc in PARKING_LUCS:
        return 'Transportation - Parking'

    if cls in ('R', 'RE'):
        if luc.startswith('51'):
            return 'Single Family Residential'
        if luc.startswith('52') or luc.startswith('53') or luc.startswith('54'):
            return 'Small Multi-Family (2-4 units)'
        if luc.startswith('55') or luc.startswith('56') or luc.startswith('57') or luc.startswith('58'):
            return 'Large Multi-Family (5+ units)'
        if luc in RES_VAC or luc.startswith('50'):
            return 'Vacant Land'
        return 'Other Residential'

    if cls in ('C', 'CE'):
        if luc in LARGE_APT_LUCS:
            return 'Large Multi-Family (5+ units)'
        if luc in SMALL_APT_LUCS:
            return 'Large Multi-Family (5+ units)'
        if luc in COMM_VAC:
            return 'Vacant Land'
        if luc in PARKING_LUCS:
            return 'Transportation - Parking'
        return 'Commercial'

    if cls in ('I', 'IE'):
        return 'Industrial'

    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf.apply(classify_parcel, axis=1)

# Override: $0 improvement → Vacant Land (except Exempt)
gdf.loc[
    (gdf['certified_tax_building'] <= 0) & (gdf['PROPERTY_CATEGORY'] != 'Exempt'),
    'PROPERTY_CATEGORY'
] = 'Vacant Land'

print('Property category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts())
print(f'\nTotal parcels: {len(gdf):,}')
print(f'Full exempt: {gdf["full_exmp"].sum():,}')
print(f'Null geometry: {gdf["geometry"].isna().sum():,}')

## Section 4 — Current Tax Model

Ohio assesses at 35% of market value. City millage: 12.70 mills (4.40 unvoted + 8.30 charter).
Source: City of Cleveland ACFR 2024, Note 10 and Statistical Section S8.

`current_tax = certified_tax_total × 0.35 × 12.70 / 1000`

In [ ]:
# Taxable assessed values (35% of post-exemption market values)
gdf['taxable_land_value'] = gdf['certified_tax_land'].clip(lower=0)
gdf['taxable_improvement_value'] = gdf['certified_tax_building'].clip(lower=0)

gdf['taxable_assessed_land'] = gdf['taxable_land_value'] * ASSESSMENT_RATIO
gdf['taxable_assessed_improvement'] = gdf['taxable_improvement_value'] * ASSESSMENT_RATIO
gdf['taxable_assessed_total'] = gdf['taxable_assessed_land'] + gdf['taxable_assessed_improvement']

gdf['millage_rate'] = CITY_MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_assessed_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

print(f'Modeled city revenue: ${current_revenue:,.0f}')
print(f'City millage: {CITY_MILLAGE} mills')
print(f'Assessment ratio: {ASSESSMENT_RATIO:.0%}')
print(f'Taxable assessed base: ${gdf["taxable_assessed_total"].sum():,.0f}')
print(f'Gross taxable market base: ${gdf["certified_tax_total"].sum():,.0f}')

## Section 5 — 4:1 Split-Rate Model

In [ ]:
taxable = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_assessed_land',
    improvement_value_col='taxable_assessed_improvement',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0

gdf = pd.concat([taxable, exempt]).sort_index()

print(f'Land millage: {land_millage:.4f} mills')
print(f'Improvement millage: {improvement_millage:.4f} mills')
print(f'Revenue check: ${new_revenue:,.0f} (target: ${taxable["current_tax"].sum():,.0f})')

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Cleveland — 4:1 Split-Rate Tax Impact')

## Section 6 — Exploration Charts

In [ ]:
plot_cats = category_summary[
    (category_summary['property_count'] >= 25)
    & (~category_summary['PROPERTY_CATEGORY'].isin(['Exempt', 'Other']))
].copy()

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#d9534f' if v > 0 else '#5cb85c' for v in plot_cats['median_tax_change_pct']]
plot_cats = plot_cats.sort_values('median_tax_change_pct')
ax.barh(plot_cats['PROPERTY_CATEGORY'], plot_cats['median_tax_change_pct'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Median % Change in Tax')
ax.set_title('Cleveland — 4:1 Split-Rate: Median Tax Change by Category')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print('Saved category_preview.png')

## Section 7 — Census Join + Export

In [ ]:
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            # census_gdf already carries demographics — spatial join added them above
            # Compute derived pct columns if not present
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
from lvt.lvt_utils import save_standard_export

out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print('Done.')